In [5]:
import pandas as pd

# Set the maximum number of columns to display (None = all columns)
pd.set_option("display.max_columns", None)

# Set the width of the display in characters (None = auto-detect screen width)
pd.set_option("display.width", 1000)

# Optional: Set the max width of a single column so text doesn't get cut off
pd.set_option("display.max_colwidth", None)

In [ ]:
# 1. Load the CSV file


file_path = r"C:\Users\ping\Desktop\grid.csv"
df = pd.read_csv(file_path)

# 2. Define the regex pattern with named groups for each hyperparameter
pattern = (
    r"model_seed_(?P<seed>\d+)_"
    r"update_(?P<update>\d+)_"
    r"ent_(?P<ent>[\d\.]+)_"
    r"pen_(?P<pen>[\d\.]+)_"
    r"lr_(?P<lr>[\d\.]+)_"
    r"bs_(?P<bs>\d+)_"
    r"steps_(?P<steps>\d+)_"
    r"envs_(?P<envs>\d+)_"
    r"ep_(?P<ep>\d+)_"
    r"sh_(?P<sh>[\d\.]+)\.pt"
)

# 3. List of target columns
param_cols = [
    "seed",
    "update",
    "ent",
    "pen",
    "lr",
    "bs",
    "steps",
    "envs",
    "ep",
    "sh",
]

# 4. Extract parameters from "Model Name" and update the DataFrame columns
extracted_df = df["Model Name"].str.extract(pattern)
df[param_cols] = extracted_df[param_cols]

# 5. Optional: Convert columns to appropriate numeric data types
numeric_type_map = {
    "seed": "int64",
    "update": "int64",
    "ent": "float64",
    "pen": "float64",
    "lr": "float64",
    "bs": "int64",
    "steps": "int64",
    "envs": "int64",
    "ep": "int64",
    "sh": "float64",
}
df = df.astype(numeric_type_map)

# 6. Save the updated DataFrame back to a CSV file
df.to_csv("grid_parsed.csv", index=False)

print("Parsing complete. Sample output:")
print(df[["Model Name"] + param_cols].head())  # Display the first few parsed rows

Parsing complete. Sample output:
                                          Model Name    seed  update    ent  \
0  "model_seed_987654_update_4_ent_0.001_pen_1.0_...  987654       4  0.001   
1  "model_seed_987654_update_4_ent_0.001_pen_1.0_...  987654       4  0.001   
2  "model_seed_987654_update_4_ent_0.001_pen_2.0_...  987654       4  0.001   
3  "model_seed_987654_update_4_ent_0.001_pen_2.0_...  987654       4  0.001   
4  "model_seed_987654_update_4_ent_0.01_pen_1.0_l...  987654       4  0.010   

   pen      lr    bs  steps  envs  ep    sh  
0  1.0  0.0001  4096    512    64  39  2.04  
1  1.0  0.0003  4096    512    64  15  1.80  
2  2.0  0.0001  4096    512    64  15  1.72  
3  2.0  0.0003  4096    512    64  42  2.02  
4  1.0  0.0001  4096    512    64  66  1.78  


In [6]:
print(df)

    Index   Date          Param Sweep                                                                                           Model Name  OOS Total Cumulative Return   OOS Sharpe Ratio (Ann)  Total Epochs Completed  Training Final approx_kl   Training Final explained_variance  Training Final avg_reward Processor Same Param    seed  update    ent  pen      lr     bs  steps  envs  ep    sh
0       1  46245  param_grid_sweep_v1  "model_seed_987654_update_4_ent_0.001_pen_1.0_lr_0.0001_bs_4096_steps_512_envs_64_ep_39_sh_2.04.pt"                        0.2796                   0.561                      75                   0.004304                           0.623800                  -0.005932    T4 GPU        NaN  987654       4  0.001  1.0  0.0001   4096    512    64  39  2.04
1       2  46245  param_grid_sweep_v1  "model_seed_987654_update_4_ent_0.001_pen_1.0_lr_0.0003_bs_4096_steps_512_envs_64_ep_15_sh_1.80.pt"                        0.3399                   0.696                      

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 22 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Index                              32 non-null     int64  
 1   Date                               32 non-null     int64  
 2   Param Sweep                        32 non-null     object 
 3   Model Name                         32 non-null     object 
 4   OOS Total Cumulative Return        32 non-null     float64
 5   OOS Sharpe Ratio (Ann)             32 non-null     float64
 6   Total Epochs Completed             32 non-null     int64  
 7   Training Final approx_kl           32 non-null     float64
 8   Training Final explained_variance  32 non-null     float64
 9   Training Final avg_reward          32 non-null     float64
 10  Processor                          32 non-null     object 
 11  Same Param                         4 non-null      object 
 

In [11]:
# Clean the column names
df.columns = df.columns.str.strip()

# Now the code will work as intended
cols_to_agg = ["OOS Total Cumulative Return", "OOS Sharpe Ratio (Ann)"]
update_4_stats = (
    df[(df["Param Sweep"] == "param_grid_sweep_v2") & (df["update"] == 4)]
    .groupby("Model Name")[cols_to_agg]
    .agg(["mean", "std"])
)

In [12]:
# Define the columns to aggregate
cols_to_agg = ["OOS Total Cumulative Return", "OOS Sharpe Ratio (Ann)"]

# [1] Create update_4_df
update_4_df = df[(df["Param Sweep"] == "param_grid_sweep_v2") & (df["update"] == 4)]

# [1a] Calculate mean and std
# (Change 'Model Name' to the column you want to group by, or remove it for global stats)
update_4_stats = update_4_df.groupby("Model Name")[cols_to_agg].agg(["mean", "std"])

# [2] Create update_8_df
update_8_df = df[(df["Param Sweep"] == "param_grid_sweep_v2") & (df["update"] == 8)]

# [2a] Calculate mean and std
update_8_stats = update_8_df.groupby("Model Name")[cols_to_agg].agg(["mean", "std"])

# Displaying results
print("--- Update 4 Stats ---")
print(update_4_stats)
print("\n--- Update 8 Stats ---")
print(update_8_stats)

--- Update 4 Stats ---
                                                                                                    OOS Total Cumulative Return     OOS Sharpe Ratio (Ann)    
                                                                                                                           mean std                   mean std
Model Name                                                                                                                                                    
"model_seed_987654_update_4_ent_0.005_pen_1.0_lr_0.0003_bs_4096_steps_512_envs_64_ep_36_sh_1.85.pt"                      0.7337 NaN                  1.283 NaN
"model_seed_987654_update_4_ent_0.005_pen_1.0_lr_0.0005_bs_4096_steps_512_envs_64_ep_36_sh_1.85.pt"                      0.5800 NaN                  0.991 NaN
"model_seed_987654_update_4_ent_0.005_pen_2.0_lr_0.0003_bs_4096_steps_512_envs_64_ep_39_sh_1.92.pt"                      0.3064 NaN                  0.669 NaN
"model_seed_987654_upda